# Working with OpenAI Remote Models

## Lab Three (Estimated Duration: ~2 hours)

This lab is **API-only**. It requires `OPENAI_API_KEY` and `OPENAI_MODEL` to be set in the notebook environment. The notebook makes real OpenAI API calls using small synthetic health and social examples.

The main learning goal is not just "make the model answer". The goal is to understand each programming layer: checking credentials, building prompts, creating request payloads, sending HTTP requests, parsing responses, logging parameters, and validating outputs.

If credentials, model access, network access, request shape, or response parsing fails, the notebook raises an error. Fix the error before continuing.


## How to Use This Notebook

- Complete `api_setup.md` before opening this notebook.
- Run cells from top to bottom.
- Use only the synthetic examples included here.
- Do not paste personal data, health records, interview transcripts, confidential institutional material, or real service-user data into prompts.
- If a cell errors, read the final line of the traceback, fix the setup or code, and rerun from the relevant earlier cell.

Each setup function is introduced in a small cell. Read the markdown before running the code: it explains what the function does and why it is needed.


## Setup Overview

The setup is split into several small cells instead of one large preamble:

1. imports and constants;
2. synthetic teaching data;
3. credential checking;
4. prompt-building functions;
5. request-building functions;
6. HTTP calling and response parsing;
7. logging and status-code helpers.

Only functions that are used later in the lab are defined.


### Setup 1: Imports and Constants

This cell imports Python standard-library modules only. No third-party OpenAI package is required.

The constants define:

- the OpenAI Responses API endpoint;
- the request timeout used for API calls;
- the allowed labels for the classification tasks;
- the system instruction sent with every request.

If a request times out, increase `OPENAI_TIMEOUT_SECONDS` and rerun the setup cells. This does not create a fallback response; it only gives the real API call more time to finish.


In [ ]:
import hashlib
import json
import os
import re
import socket
import time
import urllib.error
import urllib.request
from datetime import datetime, timezone


OPENAI_RESPONSES_URL = "https://api.openai.com/v1/responses"
OPENAI_TIMEOUT_SECONDS = 240

ALLOWED_LABELS = ["access_barrier", "housing_stress", "health_need", "other"]

system_instruction = """
You are helping with a teaching exercise about LLMs for health and social science research.
Use only the information in the supplied text.
For classification tasks, return only one allowed label unless the prompt asks for another format.
"""


### Setup 2: Synthetic Teaching Data

These examples are deliberately small and synthetic. They are shaped like public comments or de-identified service notes, not real patient or service-user records.

The labels are used later for validation. The model may disagree with them; that is part of the exercise.


In [ ]:
toy_notes = [
    {
        "id": "N001",
        "text": "Resident says the evening bus was cancelled and they missed a diabetes clinic appointment.",
        "human_label": "access_barrier",
    },
    {
        "id": "N002",
        "text": "Advice note reports rent arrears, threat of eviction, and difficulty reaching the housing office.",
        "human_label": "housing_stress",
    },
    {
        "id": "N003",
        "text": "Patient forum post asks for advice after medication side effects and trouble arranging a follow-up appointment.",
        "human_label": "health_need",
    },
]

applied_notes = [
    {
        "id": "A001",
        "text": "Public comment says the evening bus was cancelled and several residents missed diabetes clinic appointments.",
        "human_label": "access_barrier",
    },
    {
        "id": "A002",
        "text": "Community advice note reports rent arrears, threat of eviction, and difficulty reaching the housing office.",
        "human_label": "housing_stress",
    },
    {
        "id": "A003",
        "text": "Patient forum post asks for advice after medication side effects and trouble arranging a follow-up appointment.",
        "human_label": "health_need",
    },
    {
        "id": "A004",
        "text": "Local survey response praises new park lighting and weekend community activities.",
        "human_label": "other",
    },
    {
        "id": "A005",
        "text": "Public meeting note says older residents cannot afford taxis to outpatient appointments.",
        "human_label": "access_barrier",
    },
]

public_complaint_extract = """
Public feedback extract:
Several residents on the estate said the new appointment booking system is hard to use.
One carer said she tried to call the clinic three times and could not get through.
Another resident said online booking works for him, but his neighbour does not have internet access.
The group asked for a phone option, clearer opening hours, and written information in plain English.
"""

deidentified_service_note = """
Public service note:
Area=Riverside.
Issue=missed clinic appointment after bus route change.
Household also mentions rent arrears.
Requested support: transport advice and benefits referral.
No immediate safeguarding concern is stated.
"""

sensitive_training_example = """
Maria Jones called from SW1A 1AA on 12/06/2026.
Phone 07123 456789.
She missed a clinic appointment after her bus route was cancelled.
She wants advice about transport options.
"""


### Setup 3: Credential Check

`require_openai_setup()` is intentionally small and strict.

It checks two environment variables:

- `OPENAI_API_KEY`;
- `OPENAI_MODEL`.

If either is missing, it raises an error. The function returns a short key preview so learners can see that a key exists without printing the full secret.


In [ ]:
def require_openai_setup():
    missing = [name for name in ["OPENAI_API_KEY", "OPENAI_MODEL"] if not os.getenv(name)]
    if missing:
        raise EnvironmentError(
            "Missing required environment variable(s): "
            + ", ".join(missing)
            + ". Complete api_setup.md, restart the notebook kernel, and rerun this cell."
        )

    key = os.environ["OPENAI_API_KEY"]
    model = os.environ["OPENAI_MODEL"]
    return {
        "has_key": True,
        "key_preview": key[:7] + "..." + key[-4:],
        "model": model,
    }


### Setup 4: Prompt Builders

These functions turn ordinary Python strings into task-specific prompts.

Why use functions here?

- The same wording can be reused across multiple examples.
- The task instructions stay consistent.
- The notebook is easier to debug because prompt construction is separated from API calling.

`redact_for_api()` is a teaching example. It catches obvious identifiers in the synthetic note, but it is not a complete de-identification method.


In [ ]:
def make_label_prompt(text):
    return f"""
Classify the text as exactly one of:
access_barrier, housing_stress, health_need, other.

Return only the label.

Text:
{text}
""".strip()


def make_summary_prompt(text):
    return f"""
Summarise the public feedback extract in two short bullet points.
Then add one sentence beginning 'Uncertainty:' that states what we do not know.

Text:
{text}
""".strip()


def make_extraction_prompt(text):
    return f"""
Return valid JSON only with these keys:
area, main_issue, secondary_issue, requested_support, risk_level.

Use only the information in the note.

Note:
{text}
""".strip()


def redact_for_api(text):
    redacted = re.sub(r"\b[A-Z][a-z]+ [A-Z][a-z]+\b", "[NAME]", text)
    redacted = re.sub(r"\b0\d{4}\s?\d{6}\b", "[PHONE]", redacted)
    redacted = re.sub(r"\b[A-Z]{1,2}\d[A-Z\d]?\s?\d[A-Z]{2}\b", "[POSTCODE]", redacted)
    redacted = re.sub(r"\b\d{1,2}/\d{1,2}/\d{2,4}\b", "[DATE]", redacted)
    return redacted


### Setup 5: Request Builders

These functions prepare the HTTP request.

`build_openai_payload()` creates the JSON body sent to OpenAI.

`build_openai_request()` adds the URL and headers.

`redact_openai_request()` makes a safe-to-print version by hiding the API key. Always inspect the redacted request, never the full secret-bearing request.


In [ ]:
def build_openai_payload(prompt, temperature=None, top_p=None, max_output_tokens=80):
    setup = require_openai_setup()
    payload = {
        "model": setup["model"],
        "input": [
            {"role": "system", "content": system_instruction.strip()},
            {"role": "user", "content": prompt},
        ],
        "max_output_tokens": max_output_tokens,
    }
    if temperature is not None:
        payload["temperature"] = temperature
    if top_p is not None:
        payload["top_p"] = top_p
    return payload


def build_openai_request(payload):
    key = os.environ["OPENAI_API_KEY"]
    return {
        "url": OPENAI_RESPONSES_URL,
        "headers": {
            "Authorization": f"Bearer {key}",
            "Content-Type": "application/json",
        },
        "payload": payload,
    }


def redact_openai_request(request_spec):
    return {
        "url": request_spec["url"],
        "headers": {
            "Authorization": "Bearer [REDACTED]",
            "Content-Type": request_spec["headers"]["Content-Type"],
        },
        "payload": request_spec["payload"],
    }


### Setup 6: Calling OpenAI and Parsing Text

This cell contains the actual HTTP machinery.

`post_json()` sends a JSON request and raises an error for HTTP, network, or timeout failures. A timeout is still a real failure; the notebook does not substitute an answer.

`extract_text_from_openai_response()` extracts text from the OpenAI response shape and raises an error if no text is found.

`call_openai()` combines payload creation, request creation, HTTP sending, and text parsing into one function used by the exercises.

`normalise_label()` converts model text into one of the allowed labels, or `invalid` if the output does not match the label set.


In [ ]:
def post_json(url, headers, payload, timeout=None):
    if timeout is None:
        timeout = OPENAI_TIMEOUT_SECONDS

    data = json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(url, data=data, headers=headers, method="POST")
    try:
        start = time.time()
        with urllib.request.urlopen(request, timeout=timeout) as response:
            raw_body = response.read().decode("utf-8")
        elapsed = time.time() - start
        return {
            "status": response.status,
            "elapsed_seconds": elapsed,
            "json": json.loads(raw_body),
        }
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"OpenAI HTTP error {exc.code}: {body}") from exc
    except urllib.error.URLError as exc:
        reason = getattr(exc, "reason", exc)
        if isinstance(reason, (TimeoutError, socket.timeout)):
            raise TimeoutError(
                f"OpenAI request timed out after {timeout} seconds before a complete response was received. "
                "Rerun the cell, check the network, reduce max_output_tokens/batch size, "
                "or increase OPENAI_TIMEOUT_SECONDS in Setup 1."
            ) from exc
        raise ConnectionError(f"OpenAI request failed before receiving a response: {reason}") from exc
    except (TimeoutError, socket.timeout) as exc:
        raise TimeoutError(
            f"OpenAI request timed out after {timeout} seconds while waiting for the response body. "
            "Rerun the cell, check the network, reduce max_output_tokens/batch size, "
            "or increase OPENAI_TIMEOUT_SECONDS in Setup 1."
        ) from exc

def extract_text_from_openai_response(response_json):
    if response_json.get("output_text"):
        return response_json["output_text"].strip()

    chunks = []
    for item in response_json.get("output", []):
        for content in item.get("content", []):
            if isinstance(content, dict) and content.get("text"):
                chunks.append(content["text"])

    text = "\n".join(chunk.strip() for chunk in chunks if chunk.strip())
    if not text:
        raise ValueError(f"Could not find response text. Top-level keys: {list(response_json.keys())}")
    return text


def call_openai(prompt, temperature=None, top_p=None, max_output_tokens=80):
    payload = build_openai_payload(
        prompt=prompt,
        temperature=temperature,
        top_p=top_p,
        max_output_tokens=max_output_tokens,
    )
    request_spec = build_openai_request(payload)
    result = post_json(
        url=request_spec["url"],
        headers=request_spec["headers"],
        payload=request_spec["payload"],
    )
    text = extract_text_from_openai_response(result["json"])
    return {
        "status": result["status"],
        "elapsed_seconds": result["elapsed_seconds"],
        "payload": payload,
        "json": result["json"],
        "text": text,
    }


def normalise_label(output_text):
    text = (output_text or "").strip().lower()
    text = re.sub(r"^[`'\"\s]+|[`'\"\s]+$", "", text)
    for label in ALLOWED_LABELS:
        if text == label or text.startswith(label + " "):
            return label
    return "invalid"


### Setup 7: Logging and Status-Code Helpers

These helpers support reproducibility and debugging.

`prompt_hash()` stores a short fingerprint of a prompt without printing the whole prompt.

`make_log_entry()` records the model, parameters, status, and timing for a real API call.

`explain_api_status()` gives beginner-friendly interpretations of common HTTP status codes.


In [ ]:
def prompt_hash(prompt):
    return hashlib.sha256(prompt.encode("utf-8")).hexdigest()[:12]


def make_log_entry(task, prompt, result, temperature=None, top_p=None, max_output_tokens=80):
    return {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "task": task,
        "provider": "openai",
        "model": os.environ["OPENAI_MODEL"],
        "prompt_hash": prompt_hash(prompt),
        "temperature": temperature,
        "top_p": top_p,
        "max_output_tokens": max_output_tokens,
        "status": result["status"],
        "elapsed_seconds": result["elapsed_seconds"],
    }


def explain_api_status(status_code):
    explanations = {
        200: "Success: parse, validate, and log the response.",
        400: "Bad request: inspect endpoint, payload shape, model name, or parameters.",
        401: "Authentication problem: check key, billing, or project permissions.",
        403: "Permission problem: check key, billing, or model access.",
        404: "Not found: check endpoint URL or model name.",
        413: "Request too large: reduce prompt, context, files, or batch size.",
        429: "Rate limited: wait, reduce request rate, or check quota.",
        500: "Provider-side problem: retry later with logging.",
    }
    return explanations.get(status_code, "Unexpected status: inspect the response body and request log.")


## Part A: OpenAI Setup and Prompt (Approx. 20 minutes)

In this part, you check that the notebook can see your OpenAI environment variables and build the first prompt.

Keep the task small. A short classification prompt is easier to debug than a long, complex request.


### Question 1: Check OpenAI Setup

Run the setup check. It should:

- confirm that a key exists;
- print only a short preview of the key;
- print the model name.

If this cell errors, the notebook cannot see `OPENAI_API_KEY` or `OPENAI_MODEL`. Return to `api_setup.md`, restart the kernel, and rerun the setup cells.


In [ ]:
# Answer 1
setup_status = require_openai_setup()
print("OpenAI key available:", setup_status["has_key"])
print("Key preview:", setup_status["key_preview"])
print("OpenAI model:", setup_status["model"])


### Question 2: Build a Toy Prompt

Start with one synthetic note. Your aim is to build a prompt that is easy to inspect before sending.

Programming steps:

- choose the first note;
- pass the note text into `make_label_prompt(...)`;
- print the prompt and check the allowed labels are clear.


In [ ]:
# Answer 2
selected_note = toy_notes[0]
prompt = make_label_prompt(selected_note["text"])

print("Selected note:", selected_note["id"])
print(prompt)


## Part B: Build and Inspect One OpenAI Request (Approx. 25 minutes)

Do not send a request until you understand what is in it.

In this part, you build the payload, wrap it in a request spec, and inspect a redacted version that hides the API key.


### Question 3: Build the OpenAI Payload

The payload is the JSON body sent to OpenAI.

Programming steps:

- call `build_openai_payload(...)`;
- keep `max_output_tokens` small;
- print the payload with indentation;
- check the model, system instruction, user prompt, and output limit.


In [ ]:
# Answer 3
payload = build_openai_payload(
    prompt=prompt,
    temperature=None,
    top_p=None,
    max_output_tokens=30,
)

print(json.dumps(payload, indent=2))


### Question 4: Build a Redacted Request Spec

The request spec adds URL and headers. Headers contain the secret API key, so never print the raw request spec.

Programming steps:

- call `build_openai_request(...)`;
- pass that into `redact_openai_request(...)`;
- print only the redacted request.


In [ ]:
# Answer 4
request_spec = build_openai_request(payload)
redacted_request = redact_openai_request(request_spec)

print(json.dumps(redacted_request, indent=2))


## Part C: Make, Parse, and Compare OpenAI Calls (Approx. 35 minutes)

These cells make real OpenAI calls. They are intentionally small: one prompt, short output, synthetic text.

If a call fails, use the traceback and the status-code guidance later in the lab to identify the problem.


### Question 5: Make One OpenAI Call

This is the first live API call.

Programming steps:

- call `call_openai(...)`;
- store the result in `api_result`;
- print one clear confirmation line beginning `OPENAI API CALLED: YES`;
- print status, timing, and response text.


In [ ]:
# Answer 5
api_result = call_openai(
    prompt=prompt,
    temperature=None,
    top_p=None,
    max_output_tokens=30,
)

print(f"OPENAI API CALLED: YES | task=single_note_classification | model={os.environ['OPENAI_MODEL']} | status={api_result['status']} | elapsed_seconds={api_result['elapsed_seconds']:.2f}")
print("Status:", api_result["status"])
print("Elapsed seconds:", round(api_result["elapsed_seconds"], 2))
print("Text:", api_result["text"])


### Question 6: Parse Response Text

The parser should work on the real response from Question 5.

Programming steps:

- pass `api_result["json"]` into `extract_text_from_openai_response(...)`;
- normalise the text into a label;
- print both values.


In [ ]:
# Answer 6
parsed_text = extract_text_from_openai_response(api_result["json"])
parsed_label = normalise_label(parsed_text)

print("Parsed text:", parsed_text)
print("Normalised label:", parsed_label)


### Question 7: Compare Two Parameter Settings

Parameters are part of the method. Even when the prompt is the same, different parameter settings can change output.

Programming steps:

- define a small list of parameter settings;
- loop over the settings;
- call OpenAI once per setting;
- print a confirmation line for each real API call;
- store the output and normalised label.


In [ ]:
# Answer 7
parameter_tests = [
    {"name": "default_sampling", "temperature": None, "top_p": None, "max_output_tokens": 30},
    {"name": "low_temperature", "temperature": 0.0, "top_p": 1.0, "max_output_tokens": 30},
]

parameter_results = []

for setting in parameter_tests:
    result = call_openai(
        prompt=prompt,
        temperature=setting["temperature"],
        top_p=setting["top_p"],
        max_output_tokens=setting["max_output_tokens"],
    )
    print(f"OPENAI API CALLED: YES | task=parameter_comparison | setting={setting['name']} | model={os.environ['OPENAI_MODEL']} | status={result['status']} | elapsed_seconds={result['elapsed_seconds']:.2f}")
    parameter_results.append(
        {
            "name": setting["name"],
            "temperature": setting["temperature"],
            "top_p": setting["top_p"],
            "status": result["status"],
            "text": result["text"],
            "normalised_label": normalise_label(result["text"]),
        }
    )

print(json.dumps(parameter_results, indent=2))


## Part D: Logging, Failure Handling, and Validation (Approx. 25 minutes)

Real API work needs records. A useful log does not need to contain the whole prompt or API key, but it should record enough to reproduce the call.


### Question 8: Create a Reproducibility Log Entry

Create a compact log entry for the first real API call.

Programming steps:

- call `make_log_entry(...)`;
- pass the same parameters used in Question 5;
- print the log entry as JSON.


In [ ]:
# Answer 8
log_entry = make_log_entry(
    task="single_note_classification",
    prompt=prompt,
    result=api_result,
    temperature=None,
    top_p=None,
    max_output_tokens=30,
)

print(json.dumps(log_entry, indent=2))


### Question 9: Explain API Failure Codes

When a real call fails, the status code helps identify the next fix.

This question does not call the API. It prints a reference table for common status codes.


In [ ]:
# Answer 9
for status_code in [200, 400, 401, 403, 404, 413, 429, 500]:
    print(status_code, "->", explain_api_status(status_code))


### Question 10: Validate Against the Human Label

The model output is not evidence until it is checked.

Programming steps:

- compare the model label with the human label;
- store the comparison in a dictionary;
- print the result.


In [ ]:
# Answer 10
validation_result = {
    "note_id": selected_note["id"],
    "human_label": selected_note["human_label"],
    "model_text": api_result["text"],
    "model_label": normalise_label(api_result["text"]),
}
validation_result["match"] = validation_result["model_label"] == validation_result["human_label"]

print(json.dumps(validation_result, indent=2))


## Part E: Applied Health and Social Workflows (Approx. 45 minutes)

Now apply the same real API pattern to four small tasks:

- batch classification;
- summarisation;
- structured extraction;
- redaction before sending text.

The examples remain synthetic and small, but the programming pattern is the same pattern used in larger workflows.


### Programming Pattern for the Applied Tasks

Each applied task follows the same sequence:

1. Put text in a variable.
2. Build a clear prompt.
3. Call OpenAI.
4. Parse the response.
5. Validate, inspect, or log the output.

When debugging, reduce the workflow to one row, one prompt, and one printed response before scaling back up.


### Question 11: Batch Classify Applied Notes

Batch work means repeating the same operation over several examples.

Programming steps:

- loop over `applied_notes`;
- build one label prompt per note;
- call OpenAI once per note;
- normalise the output;
- print a confirmation line for each real API call;
- compare with the human label.


In [ ]:
# Answer 11
batch_rows = []

for note in applied_notes:
    print(f"OPENAI API CALL STARTING: task=batch_classification | note_id={note['id']}")
    note_prompt = make_label_prompt(note["text"])
    result = call_openai(
        prompt=note_prompt,
        temperature=None,
        top_p=None,
        max_output_tokens=20,
    )
    print(f"OPENAI API CALLED: YES | task=batch_classification | note_id={note['id']} | model={os.environ['OPENAI_MODEL']} | status={result['status']} | elapsed_seconds={result['elapsed_seconds']:.2f}")
    model_label = normalise_label(result["text"])
    batch_rows.append(
        {
            "id": note["id"],
            "human_label": note["human_label"],
            "model_text": result["text"],
            "model_label": model_label,
            "match": model_label == note["human_label"],
            "status": result["status"],
            "elapsed_seconds": result["elapsed_seconds"],
        }
    )

for row in batch_rows:
    print(row)


### Question 12: Evaluate the Batch Classifier

A classifier should be evaluated as a classifier, not just inspected row by row.

Use the `batch_rows` object from Question 11 to calculate:

- overall accuracy;
- how many outputs were outside the allowed label set;
- how many examples were tested per human label;
- which rows disagreed with the human label.

The cell also prints a single-line accuracy summary so the main classifier result is easy to see.

This is still a tiny synthetic evaluation set, so the numbers are not research evidence. The point is to practise the evaluation structure.


In [ ]:
# Answer 12
total_rows = len(batch_rows)
correct_rows = 0
invalid_rows = 0
per_label_counts = {}
disagreement_rows = []

for row in batch_rows:
    if row["match"]:
        correct_rows = correct_rows + 1
    if row["model_label"] not in ALLOWED_LABELS:
        invalid_rows = invalid_rows + 1

    human_label = row["human_label"]
    per_label_counts[human_label] = per_label_counts.get(human_label, 0) + 1

    if not row["match"]:
        disagreement_rows.append(row)

accuracy = correct_rows / total_rows

classifier_evaluation = {
    "n": total_rows,
    "accuracy": accuracy,
    "invalid_outputs": invalid_rows,
    "per_label_counts": per_label_counts,
    "disagreements": disagreement_rows,
}

print(f"CLASSIFIER ACCURACY ACROSS ALL BATCH EXAMPLES: {correct_rows}/{total_rows} = {accuracy:.3f}")
print(f"CLASSIFIER INVALID OUTPUTS ACROSS ALL BATCH EXAMPLES: {invalid_rows}/{total_rows}")
print(json.dumps(classifier_evaluation, indent=2))


### Question 13: Summarise a Public Complaint Extract

Summarisation is different from classification. The output is prose, so validation requires reading for accuracy and uncertainty.

Programming steps:

- build a summary prompt;
- call OpenAI with a slightly larger output limit;
- print the summary.


In [ ]:
# Answer 13
summary_prompt = make_summary_prompt(public_complaint_extract)
summary_result = call_openai(
    prompt=summary_prompt,
    temperature=None,
    top_p=None,
    max_output_tokens=160,
)

print(f"OPENAI API CALLED: YES | task=summarisation | model={os.environ['OPENAI_MODEL']} | status={summary_result['status']} | elapsed_seconds={summary_result['elapsed_seconds']:.2f}")
summary_text = summary_result["text"]
print(summary_text)


### Question 14: Extract Structured Fields

Structured extraction is useful because the output can be parsed into a Python dictionary.

Programming steps:

- ask for valid JSON only;
- call OpenAI;
- parse the returned text with `json.loads`;
- inspect the keys.

If parsing fails, the prompt or model output needs attention.


In [ ]:
# Answer 14
extraction_prompt = make_extraction_prompt(deidentified_service_note)
extraction_result = call_openai(
    prompt=extraction_prompt,
    temperature=None,
    top_p=None,
    max_output_tokens=180,
)

print(f"OPENAI API CALLED: YES | task=structured_extraction | model={os.environ['OPENAI_MODEL']} | status={extraction_result['status']} | elapsed_seconds={extraction_result['elapsed_seconds']:.2f}")
extracted_fields = json.loads(extraction_result["text"])
print(json.dumps(extracted_fields, indent=2))


### Question 15: Minimise and Redact Before Sending Text

Before sending text to a hosted model, ask whether each detail is needed.

Programming steps:

- redact obvious identifiers from the synthetic note;
- build a classification prompt from the redacted note;
- call OpenAI;
- print the redacted note and model label.


In [ ]:
# Answer 15
redacted_note = redact_for_api(sensitive_training_example)
redacted_prompt = make_label_prompt(redacted_note)

redacted_result = call_openai(
    prompt=redacted_prompt,
    temperature=None,
    top_p=None,
    max_output_tokens=30,
)

print(f"OPENAI API CALLED: YES | task=redacted_note_classification | model={os.environ['OPENAI_MODEL']} | status={redacted_result['status']} | elapsed_seconds={redacted_result['elapsed_seconds']:.2f}")
print("Redacted note:")
print(redacted_note.strip())
print("\nModel text:", redacted_result["text"])
print("Model label:", normalise_label(redacted_result["text"]))


### Question 16: Final Mini-Case

Finish by documenting one responsible OpenAI workflow.

Your report should include:

- the task;
- the model;
- privacy step;
- parameters;
- validation evidence;
- likely failure modes.


In [ ]:
# Answer 16
mini_case = {
    "task": "classify short public comments",
    "provider_and_model": f"OpenAI / {os.environ['OPENAI_MODEL']}",
    "privacy_step": "use synthetic or approved de-identified text; redact direct identifiers before sending",
    "parameters": {
        "temperature": None,
        "top_p": None,
        "max_output_tokens": 30,
    },
    "validation": "compare model labels with human labels and inspect disagreements",
    "evidence_from_lab": {
        "single_note_validation": validation_result,
        "batch_rows": batch_rows,
        "classifier_evaluation": classifier_evaluation,
        "classifier_accuracy": classifier_evaluation["accuracy"],
    },
}

print(json.dumps(mini_case, indent=2))


## End of Lab Checklist

By the end of this lab, you should have:

- checked `OPENAI_API_KEY` and `OPENAI_MODEL`;
- understood what each setup helper function does;
- built an OpenAI request payload;
- inspected a redacted request before sending it;
- made real OpenAI API calls;
- printed explicit OpenAI API call confirmation lines;
- parsed real response text;
- compared two parameter settings;
- created a reproducibility log entry;
- interpreted common API status codes;
- compared output with human labels;
- evaluated batch-classifier accuracy and disagreements;
- printed a single-line classifier accuracy summary;
- batch-classified short health and social notes;
- summarised a public feedback extract;
- extracted structured fields from a de-identified note;
- redacted obvious identifiers before sending text;
- described failure modes for a small applied workflow.
